<a href="https://colab.research.google.com/github/lygitdata/GarmentIQ/blob/main/test/adv_usage_classification_model_training_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Usage - GarmentIQ Classification Model Training and Evaluation

Fine-tuning adapts an existing model, but sometimes you need to train one from scratch.
GarmentIQ can train any PyTorch model through the same interface, whether it is one of the
predefined architectures or your own.

This tutorial shows how to train the built-in CNN3, how to plug in a model you wrote
yourself, and how to evaluate both on a held-out test set so the results can be compared
fairly.

## Table of Contents

1. [Prerequisites](#prerequisites)
2. [Prepare the data](#data)
3. [Train a predefined model](#cnn3)
4. [Train your own model](#custom)
5. [Evaluate and compare](#evaluate)

<a name="prerequisites"></a>
## Prerequisites

Install the package and download the training data. On Colab you can keep this section
collapsed.

> **Your data must be a zip file with the same structure as ours**, that is an image
> folder plus a `metadata.csv` naming each file and its label. See
> [the example dataset](https://www.kaggle.com/datasets/lygitdata/garmentiq-classification-set-nordstrom-and-myntra)
> for the exact layout.

In [ ]:
# @title Install GarmentIQ
!pip install garmentiq -q

In [ ]:
# @title Import GarmentIQ and choose a device

import torch
import torch.nn as nn
import torch.optim as optim

import garmentiq as giq
from garmentiq.classification.model_definition import CNN3
from garmentiq.classification.utils import CachedDataset

# GarmentIQ never grabs an accelerator on its own. For training the device is passed
# inside `param`, and it defaults to "cpu".
# Both models below end in an adaptive pooling layer whose input size is not divisible
# by its output size, which PyTorch does not implement on Apple Silicon ("mps"), so
# this notebook uses CUDA or CPU. tinyViT has no such layer and does run on "mps";
# see the fine-tuning notebook.
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
# @title Download the training data

# About 1.5 GB
!curl -sL -o garmentiq-classification-set-nordstrom-and-myntra.zip \
  https://www.kaggle.com/api/v1/datasets/download/lygitdata/garmentiq-classification-set-nordstrom-and-myntra

print("Download finished.")

<a name="data"></a>
## Prepare the data

Unlike fine-tuning, training from scratch needs a held-out test set, so 15% is reserved
here. Both models below are trained and evaluated on exactly the same split, which is what
makes the comparison meaningful.

In [ ]:
data = giq.classification.train_test_split(
    output_dir="data",
    train_zip_dir="garmentiq-classification-set-nordstrom-and-myntra.zip",
    metadata_csv="metadata.csv",
    label_column="garment",
    test_size=0.15,
    verbose=True,
)

These models are trained at a smaller `resize_dim` than the shipped
tinyViT, which trains faster and suits the simpler architectures. Whatever you choose here
must be reused at evaluation and prediction time.

In [ ]:
train_images, train_labels, _ = giq.classification.load_data(
    df=data["train_metadata"],
    img_dir=data["train_images"],
    label_column="garment",
    resize_dim=(60, 92),
    normalize_mean=[0.8047, 0.7808, 0.7769],
    normalize_std=[0.2957, 0.3077, 0.3081],
)

<a name="cnn3"></a>
## Train a predefined model

`train_pytorch_nn` takes the architecture as `model_class` and its constructor arguments
as `model_args`, so switching models is a one-line change. Training is cross-validated and
the checkpoint with the lowest cross-entropy is kept as the best one.

Two folds and five epochs are used here for demonstration.

In [ ]:
giq.classification.train_pytorch_nn(
    model_class=CNN3,
    model_args={"num_classes": 9},
    dataset_class=CachedDataset,
    dataset_args={
        "metadata_df": data["train_metadata"],
        "raw_labels": data["train_metadata"]["garment"],
        "cached_images": train_images,
        "cached_labels": train_labels,
    },
    param={
        "optimizer_class": optim.AdamW,
        "optimizer_args": {"lr": 0.001, "weight_decay": 1e-4},
        "n_fold": 2,
        "n_epoch": 5,
        "patience": 2,
        "batch_size": 256,
        "model_save_dir": "cnn3_models",
        "best_model_name": "best_cnn3_model.pt",
        "device": device,
    },
)

<a name="custom"></a>
## Train your own model

Any `torch.nn.Module` works. GarmentIQ only requires that the constructor accepts the
arguments you list in `model_args`, and that `forward` returns one logit per class.

In [ ]:
class UserDefinedCNN(nn.Module):
    def __init__(self, num_classes):
        super(UserDefinedCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

The training call is identical apart from `model_class` and the output
directory.

In [ ]:
giq.classification.train_pytorch_nn(
    model_class=UserDefinedCNN,
    model_args={"num_classes": 9},
    dataset_class=CachedDataset,
    dataset_args={
        "metadata_df": data["train_metadata"],
        "raw_labels": data["train_metadata"]["garment"],
        "cached_images": train_images,
        "cached_labels": train_labels,
    },
    param={
        "optimizer_class": optim.AdamW,
        "optimizer_args": {"lr": 0.001, "weight_decay": 1e-4},
        "n_fold": 2,
        "n_epoch": 5,
        "patience": 2,
        "batch_size": 256,
        "model_save_dir": "userdefined_cnn_models",
        "best_model_name": "best_userdefined_cnn_model.pt",
        "device": device,
    },
)

<a name="evaluate"></a>
## Evaluate and compare

The test set was held out from both training runs, so neither model has seen it. Load it
with the same `resize_dim` and normalization used for training.

In [ ]:
test_images, test_labels, _ = giq.classification.load_data(
    df=data["test_metadata"],
    img_dir=data["test_images"],
    label_column="garment",
    resize_dim=(60, 92),
    normalize_mean=[0.8047, 0.7808, 0.7769],
    normalize_std=[0.2957, 0.3077, 0.3081],
)

In [ ]:
# The predefined CNN3
giq.classification.test_pytorch_nn(
    model_path="cnn3_models/best_cnn3_model.pt",
    model_class=CNN3,
    model_args={"num_classes": 9},
    dataset_class=CachedDataset,
    dataset_args={
        "raw_labels": data["test_metadata"]["garment"],
        "cached_images": test_images,
        "cached_labels": test_labels,
    },
    param={"batch_size": 64, "device": device},
)

In [ ]:
# The user-defined model, scored the same way
giq.classification.test_pytorch_nn(
    model_path="userdefined_cnn_models/best_userdefined_cnn_model.pt",
    model_class=UserDefinedCNN,
    model_args={"num_classes": 9},
    dataset_class=CachedDataset,
    dataset_args={
        "raw_labels": data["test_metadata"]["garment"],
        "cached_images": test_images,
        "cached_labels": test_labels,
    },
    param={"batch_size": 64, "device": device},
)

Compare the accuracy and F1 scores above to decide which architecture
suits your data. With these settings CNN3 typically wins, but the point is that both were
trained and scored through the same interface.

If you would rather adapt the shipped model than train one from scratch, see the
[fine-tuning notebook](https://colab.research.google.com/github/lygitdata/GarmentIQ/blob/main/test/adv_usage_classification_model_fine_tuning.ipynb).